In [1]:
# CELL 1 STUDENT CONFIGURATION
# Path is used for file and folder paths
from pathlib import Path
 
# Student-specific information (REPLACE the ID with your own)
LAST_NAME = "Endaya"
STUDENT_ID = "TUPM-26-4330" 
 
# Seed digit = last digit of the student ID
SEED_DIGIT = int(STUDENT_ID[-1])
# ID sum = sum of every digit in the student ID
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
 
# Display the derived values
print("Student:", LAST_NAME, STUDENT_ID)
print("Seed digit:", SEED_DIGIT)
print("ID sum:", ID_SUM)

Student: Endaya TUPM-26-4330
Seed digit: 0
ID sum: 18


In [2]:
# CELL 2 CREATE WORKSPACE FOLDERS
# Define the folders (put the notebook inside Lab6_File_Handling_Additional_Exercises)
WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"        # raw Kaggle CSV goes here (NOT uploaded to GitHub)
OUTPUT_DIR = WORK_DIR / "output"    # generated files
BACKUP_DIR = WORK_DIR / "backup"    # backups
 
# Create the folders if they do not exist yet
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)
 
# Verify the workspace
print("Workspace:", WORK_DIR.resolve())
print("Data folder exists:", DATA_DIR.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Backup folder exists:", BACKUP_DIR.exists())

Workspace: C:\Users\WINDOWS\Lab6_File_Handling_Additional_Exercise
Data folder exists: True
Output folder exists: True
Backup folder exists: True


In [3]:
# CELL 3 EXERCISE 2 PARAMETERS
# Built-in modules only (no Pandas or NumPy)
import csv
import json
import shutil
 
# Dataset location (download user_behavior_dataset.csv from Kaggle into the data folder)
dataset_path = DATA_DIR / "user_behavior_dataset.csv"
 
# Student-specific rules
target_os = "Android" if SEED_DIGIT % 2 == 0 else "iOS"   # even = Android, odd = iOS
target_class = (SEED_DIGIT % 5) + 1                       # 1 to 5
 
# Fields that will be exported
export_fields = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)",
]
 
print("Target operating system:", target_os)
print("Target behavior class:", target_class)

Target operating system: Android
Target behavior class: 1


In [4]:
# CELL 4 READ, FILTER AND AVERAGE
total_records = 0     # all data rows in the dataset
os_matches = 0        # rows matching the operating system
invalid = 0           # rows with unreadable numbers
selected = []         # rows in the target class (exported)
battery_total = 0.0
data_total = 0.0
 
if not dataset_path.exists():
    print("Dataset not found:", dataset_path)
else:
    with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:
        for row in csv.DictReader(file):
            total_records += 1
 
            # Step 1: match the operating system
            if (row.get("Operating System") or "").strip().lower() != target_os.lower():
                continue
            os_matches += 1
 
            # Step 2: match the behavior class (skip invalid values)
            try:
                behavior_class = int(row["User Behavior Class"])
                battery = float(row["Battery Drain (mAh/day)"])
                data_usage = float(row["Data Usage (MB/day)"])
            except (KeyError, TypeError, ValueError):
                invalid += 1
                continue
 
            if behavior_class != target_class:
                continue
 
            # Keep the row and add to the running totals
            selected.append({name: row[name] for name in export_fields})
            battery_total += battery
            data_total += data_usage
 
# Averages (0 when there are no matching records)
if selected:
    avg_battery = battery_total / len(selected)
    avg_data = data_total / len(selected)
else:
    avg_battery = 0.0
    avg_data = 0.0
    print("No matching records found.")
 
print("Total records:", total_records)
print("Records matching OS:", os_matches)
print("Records in target class:", len(selected))
print("Invalid records skipped:", invalid)
print("Average battery drain (mAh/day):", round(avg_battery, 2))
print("Average data usage (MB/day):", round(avg_data, 2))

Dataset not found: data\user_behavior_dataset.csv
No matching records found.
Total records: 0
Records matching OS: 0
Records in target class: 0
Invalid records skipped: 0
Average battery drain (mAh/day): 0.0
Average data usage (MB/day): 0.0


In [5]:
# CELL 5 EXPORT SELECTED USERS TO CSV
# Path of the exported CSV
csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_selected_users.csv"
 
# Write only the specified fields
with open(csv_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=export_fields)
    writer.writeheader()
    writer.writerows(selected)
 
print("Created:", csv_path)

Created: output\Endaya_TUPM-26-4330_selected_users.csv


In [6]:
# CELL 6 WRITE JSON SUMMARY AND VERIFY ROW COUNT
# Count the data rows actually written in the CSV (total lines minus the header)
with open(csv_path, "r", encoding="utf-8") as file:
    exported_rows = sum(1 for _ in file) - 1
 
# The exported row count must match the selected record count
count_check = "MATCH" if exported_rows == len(selected) else "MISMATCH"
 
# Path and content of the JSON summary
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_usage_summary.json"
summary = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "operating_system": target_os,
    "behavior_class": target_class,
    "total_records": total_records,
    "records_matching_os": os_matches,
    "records_in_target_class": len(selected),
    "average_battery_drain_mAh_day": round(avg_battery, 2),
    "average_data_usage_MB_day": round(avg_data, 2),
    "exported_rows": exported_rows,
    "count_verification": count_check,
}
 
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)
 
print("Created:", json_path)
print("Selected:", len(selected), "| Exported:", exported_rows, "->", count_check)

Created: output\Endaya_TUPM-26-4330_usage_summary.json
Selected: 0 | Exported: 0 -> MATCH


In [7]:
# CELL 7 BACKUP THE CSV AND REPORT ITS SIZE
# Copy the exported CSV into the backup folder
backup_path = BACKUP_DIR / csv_path.name
shutil.copy2(csv_path, backup_path)
 
# Report the backup file size
backup_size = backup_path.stat().st_size
 
print("Backup:", backup_path)
print("Backup exists:", backup_path.exists())
print("Backup size:", backup_size, "bytes")

Backup: backup\Endaya_TUPM-26-4330_selected_users.csv
Backup exists: True
Backup size: 83 bytes


In [8]:
# CELL 8 FINAL OUTPUT (ASSESSMENT DATA)
print("Operating system and behavior class:", target_os, "/ class", target_class)
print("Total records in dataset:", total_records)
print("Records matching operating system:", os_matches)
print("Records in target behavior class:", len(selected))
print("Average battery drain (mAh/day):", round(avg_battery, 2))
print("Average data usage (MB/day):", round(avg_data, 2))
print("Count verification:", count_check)
print("Exported CSV file:", csv_path.name)
print("JSON summary file:", json_path.name)
print("Backup file and size:", backup_path.name, backup_size, "bytes")

Operating system and behavior class: Android / class 1
Total records in dataset: 0
Records matching operating system: 0
Records in target behavior class: 0
Average battery drain (mAh/day): 0.0
Average data usage (MB/day): 0.0
Count verification: MATCH
Exported CSV file: Endaya_TUPM-26-4330_selected_users.csv
JSON summary file: Endaya_TUPM-26-4330_usage_summary.json
Backup file and size: Endaya_TUPM-26-4330_selected_users.csv 83 bytes
